---
title: "单调队列"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
from collections import deque

## 📝 题目 239：滑动窗口最大值

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `nums`，有一个大小为 `k` 的滑动窗口从数组的最左侧移动到数组的最右侧。你只可以看到在滑动窗口内的 `k` 个数字。滑动窗口每次只向右移动一位。

返回 **滑动窗口中的最大值** 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [1,3,-1,-3,5,3,6,7]`, `k = 3`
    * **输出**：`[3,3,5,5,6,7]`
    * **解释**：
      滑动窗口的位置                最大值
      ---------------               -----
      [1  3  -1] -3  5  3  6  7       3
       1 [3  -1  -3] 5  3  6  7       3
       1  3 [-1  -3  5] 3  6  7       5
       1  3  -1 [-3  5  3] 6  7       5
       1  3  -1  -3 [5  3  6] 7       6
       1  3  -1  -3  5 [3  6  7]      7

* **示例 2**：
    * **输入**：`nums = [1]`, `k = 1`
    * **输出**：`[1]`
:::

In [2]:
class Solution239:
    def maxSlidingWindow(self, nums: list[int], k: int) -> list[int]:
        q = deque()  # 双端队列，存储的是元素的 索引 (Index)，方便判断是否过期
        res = []
        for i, num in enumerate(nums):
            if q and q[0] <= i - k:  # 1. 如果队首大佬的索引已经滑出了窗口 [i-k+1, i]，必须退位
                q.popleft()
            while q and nums[q[-1]] <= num:  # 2. 新来的 num 如果比队尾的元素大
                q.pop()
            q.append(i)  # 新人入队
            if i >= k - 1:  # 3. 只要窗口已经形成 (i >= k - 1)，队首永远是当前窗口的最大值
                res.append(nums[q[0]])
        return res

In [3]:
#| code-fold: true
def test_239(func):
    test_cases = [
        {"desc": "1. 经典过山车数据", "nums": [1,3,-1,-3,5,3,6,7], "k": 3, "expected": [3,3,5,5,6,7]},
        {"desc": "2. 只有一个元素", "nums": [1], "k": 1, "expected": [1]},
        {"desc": "3. 严格递减序列 (全员保送)", "nums": [9,8,7,6,5,4], "k": 2, "expected": [9,8,7,6,5]},
        {"desc": "4. 严格递增序列 (疯狂踢人)", "nums": [1,2,3,4,5,6], "k": 3, "expected": [3,4,5,6]},
        {"desc": "5. 窗口大小等于数组大小", "nums": [4,2,7,5,8,1], "k": 6, "expected": [8]},
        {"desc": "6. 重复元素大乱斗", "nums": [3,3,3,3,2,4,2], "k": 3, "expected": [3,3,3,4,4]},
        {"desc": "7. 负数混合压测", "nums": [-7,-8,7,5,7,1,6,0], "k": 4, "expected": [7,7,7,7,7]}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<15} | {'实际':<15} | {'描述'}")
    print("-" * 80)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1

        # 截断长数组输出以便对齐
        expected_str = str(tc['expected'])[:15] + "..." if len(str(tc['expected'])) > 15 else str(tc['expected'])
        actual_str = str(actual)[:15] + "..." if len(str(actual)) > 15 else str(actual)

        print(f"{i+1:<3} | {status:<6} | {expected_str:<15} | {actual_str:<15} | {tc['desc']}")

    print("-" * 80)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_239(Solution239().maxSlidingWindow)

ID  | 结果     | 预期              | 实际              | 描述
--------------------------------------------------------------------------------
1   | ✅ PASS | [3, 3, 5, 5, 6,... | [3, 3, 5, 5, 6,... | 1. 经典过山车数据
2   | ✅ PASS | [1]             | [1]             | 2. 只有一个元素
3   | ✅ PASS | [9, 8, 7, 6, 5] | [9, 8, 7, 6, 5] | 3. 严格递减序列 (全员保送)
4   | ✅ PASS | [3, 4, 5, 6]    | [3, 4, 5, 6]    | 4. 严格递增序列 (疯狂踢人)
5   | ✅ PASS | [8]             | [8]             | 5. 窗口大小等于数组大小
6   | ✅ PASS | [3, 3, 3, 4, 4] | [3, 3, 3, 4, 4] | 6. 重复元素大乱斗
7   | ✅ PASS | [7, 7, 7, 7, 7] | [7, 7, 7, 7, 7] | 7. 负数混合压测
--------------------------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

如果你用暴力解法，每次窗口滑动都去遍历一遍窗口找最大值，时间复杂度是 $O(N \times K)$，在 Hard 题的庞大数据量下面绝对会 TLE（超时）。

**单调队列的降维打击：**
我们需要维护一个**双端队列（Deque）**。里面存放的不是具体的数字，而是数字的 **索引（Index）**（因为我们需要根据索引判断数字有没有过期）。

1. **入队规则（年轻气盛，踢出弱者）**：
   当新元素 `nums[i]` 准备进入队尾时，它会看看队尾的元素。如果队尾元素**小于等于** `nums[i]`，那么队尾元素就彻底没用了（它不仅比新元素小，而且比新元素更早离开窗口，它绝对不可能是未来的最大值）。直接把它弹出！直到队尾元素比新元素大，新元素才乖乖排在后面。这保证了队列的值永远是**单调递减**的。
2. **出队规则（英雄迟暮，被迫退位）**：
   队首永远是当前队列里的最大值。但是，每次窗口滑动时，必须检查队首的索引。如果队首的索引已经不在当前窗口内（即 `队首索引 <= i - k`），说明这个大佬已经“过期”了，必须从队首把它踢出去！
3. **收割结果**：
   经过上面两步，队首元素绝对是当前窗口的真正老大。只要当前窗口已经完全进入数组（即 `i >= k - 1`），就把队首对应的数字存入结果数组。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。这是面试官必考的痛点。虽然外层有 `for` 循环，内层有 `while` 循环，但请注意：**数组中的每一个元素，最多只会入队一次，出队一次。** 没有任何元素会被反复操作。所以均摊下来的时间复杂度是绝对的 $O(N)$。
* **空间复杂度**：$O(K)$。双端队列中最多只会存下窗口大小 `K` 个元素的索引。
:::

## 📝 题目 1438：绝对差不超过限制的最长连续子数组

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `nums` ，和一个表示限制的整数 `limit`。
请你返回最长连续子数组的长度，该子数组中的任意两个元素之间的绝对差必须小于或者等于 `limit` 。

**注意**：如果不存在满足条件的子数组，则返回 0 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [8,2,4,7]`, `limit = 4`
    * **输出**：`2`
    * **解释**：所有子数组如下：
      [8] 最大绝对差 0 <= 4.
      [8,2] 最大绝对差 |8-2| = 6 > 4.
      [2] 最大绝对差 0 <= 4.
      [2,4] 最大绝对差 |2-4| = 2 <= 4.
      [2,4,7] 最大绝对差 |2-7| = 5 > 4.
      [4,7] 最大绝对差 |4-7| = 3 <= 4.
      因此，最长子数组的长度是 2 ([2,4] 或 [4,7])。

* **示例 2**：
    * **输入**：`nums = [10,1,2,4,7,2]`, `limit = 5`
    * **输出**：`4`
:::

In [4]:
class Solution1438:
    def longestSubarray(self, nums: list[int], limit: int) -> int:
        max_q = deque() # 单调递减，维护最大值
        min_q = deque() # 单调递增，维护最小值
        left = 0
        ans = 0

        for right, num in enumerate(nums):
            # 1. 更新单调递减队列
            while max_q and max_q[-1] < num:
                max_q.pop()
            max_q.append(num)

            # 2. 更新单调递增队列
            while min_q and min_q[-1] > num:
                min_q.pop()
            min_q.append(num)

            # 3. 如果当前窗口极差超过 limit，收缩左边界
            while max_q and min_q and max_q[0] - min_q[0] > limit:
                if nums[left] == max_q[0]:
                    max_q.popleft()
                if nums[left] == min_q[0]:
                    min_q.popleft()
                left += 1

            # 4. 统计当前合法窗口的最大长度
            ans = max(ans, right - left + 1)

        return ans

In [6]:
#| code-fold: true
def test_1438(func):
    test_cases = [
        {"desc": "1. 示例 1: 常规波动", "nums": [8,2,4,7], "limit": 4, "expected": 2},
        {"desc": "2. 示例 2: 较长合规子串", "nums": [10,1,2,4,7,2], "limit": 5, "expected": 4},
        {"desc": "3. 极限: limit 为 0 (找最长重复数字)", "nums": [1,2,2,2,3], "limit": 0, "expected": 3},
        {"desc": "4. 边界: 单个元素", "nums": [10], "limit": 5, "expected": 1},
        {"desc": "5. 全员合规 (极大 limit)", "nums": [1,5,10,15], "limit": 20, "expected": 4},
        {"desc": "6. 全员不合规 (除了单节点)", "nums": [1,10,20,30], "limit": 5, "expected": 1},
        {"desc": "7. 锯齿状波动", "nums": [4,8,2,10,6], "limit": 6, "expected": 3}, # [8,2,4] 或 [10,6] -> 注意 [8,2,4] 最大差是 6
        {"desc": "8. 大数据量单调递增", "nums": list(range(100)), "limit": 10, "expected": 11}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"], tc["limit"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<4} | {str(actual):<4} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1438(Solution1438().longestSubarray)

ID  | 结果     | 预期   | 实际   | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 2    | 2    | 1. 示例 1: 常规波动
2   | ✅ PASS | 4    | 4    | 2. 示例 2: 较长合规子串
3   | ✅ PASS | 3    | 3    | 3. 极限: limit 为 0 (找最长重复数字)
4   | ✅ PASS | 1    | 1    | 4. 边界: 单个元素
5   | ✅ PASS | 4    | 4    | 5. 全员合规 (极大 limit)
6   | ✅ PASS | 1    | 1    | 6. 全员不合规 (除了单节点)
7   | ✅ PASS | 3    | 3    | 7. 锯齿状波动
8   | ✅ PASS | 11   | 11   | 8. 大数据量单调递增
---------------------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

这道题的本质是：维护一个窗口 `[left, right]`，使得该窗口内的 `max(window) - min(window) <= limit`。

**为什么需要单调队列？**
如果每次移动窗口都用 `max()` 和 `min()` 函数，时间复杂度会退化到 $O(N^2)$。我们需要在 $O(1)$ 时间内知道窗口里的“最强者”和“最弱者”。

1. **双剑合璧**：
   - **递减队列 `max_q`**：队首永远是当前窗口的最大值。
   - **递增队列 `min_q`**：队首永远是当前窗口的最小值。
2. **右边界扩张**：
   - `right` 指针每走一步，就按照单调队列的“踢人规则”将新元素分别塞进两个队列。
3. **左边界收缩（重点）**：
   - 检查当前两队的队首：`max_q[0] - min_q[0]`。
   - 如果这个差值 **> limit**，说明当前窗口太混乱了，必须缩小。
   - 增加 `left` 指针，并检查两队的队首是否是刚刚滑出的 `nums[left]`。如果是，则将其弹出。
4. **更新长度**：
   - 在每一步保证合法的窗口中，记录 `right - left + 1` 的最大值。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。每个元素最多进出两个队列各一次。所有的 `while` 循环加起来也是线性的。
* **空间复杂度**：$O(N)$。最坏情况下队列会存储所有元素的索引。
:::

## 📝 题目 862：和至少为 K 的最短子数组

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `nums` 和一个整数 `k` ，找出 `nums` 中和至少为 `k` 的 **最短非空子数组** ，并返回该子数组的长度。如果不存在这样的 **子数组** ，返回 `-1` 。

**⚠️ 终极死线**：
数组中**存在负数**！时间复杂度要求严格为 $O(N)$。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [1], k = 1`
    * **输出**：`1`

* **示例 2**：
    * **输入**：`nums = [1,2], k = 4`
    * **输出**：`-1`

* **示例 3**：
    * **输入**：`nums = [2,-1,2], k = 3`
    * **输出**：`3`
    * **解释**：[2]和为2，[2,-1]和为1，[2,-1,2]和为3。最短满足 >= 3 的是整个数组，长度 3。
:::

In [7]:
class Solution862:
    def shortestSubarray(self, nums: list[int], k: int) -> int:
        n = len(nums)
        # 1. 构建前缀和数组 (长度为 n+1，包含虚拟原点 0)
        P = [0] * (n + 1)
        for i in range(n):
            P[i + 1] = P[i] + nums[i]

        ans = float('inf')
        q = deque()  # 存储前缀和的索引

        for y, py in enumerate(P):
            # 规则一：功成身退。只要队首满足条件，记录答案，并将其踢出（因为它不可能再组成更短的了）
            while q and py - P[q[0]] >= k:
                ans = min(ans, y - q.popleft())

            # 规则二：降维打击。如果当前前缀和更小，它不仅更容易满足条件，而且距离未来更近
            while q and P[q[-1]] >= py:  # 队尾的旧元素被彻底碾压，直接踢出，保持队列单调递增
                q.pop()
            q.append(y)

        return ans if ans != float('inf') else -1

In [8]:
#| code-fold: true
def test_862(func):
    test_cases = [
        {"desc": "1. 示例 1: 基础正数", "nums": [1], "k": 1, "expected": 1},
        {"desc": "2. 示例 2: 无法满足", "nums": [1,2], "k": 4, "expected": -1},
        {"desc": "3. 示例 3: 经典负数回退", "nums": [2,-1,2], "k": 3, "expected": 3},
        {"desc": "4. 单个大数直接秒杀", "nums": [1, 2, 3, 100, 4, 5], "k": 99, "expected": 1},
        {"desc": "5. 剧烈波动的负数", "nums": [84,-37,32,40,95], "k": 167, "expected": 3},
        {"desc": "6. 全局负数", "nums": [-1, -2, -3], "k": 1, "expected": -1},
        {"desc": "7. 前方大量无用累加", "nums": [1,1,1,1,1,1,10], "k": 10, "expected": 1},
        {"desc": "8. 连续极长数组带负数断层", "nums": [2, -1, 2, -1, 2, -1, 2], "k": 4, "expected": 5}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 80)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<4} | {str(actual):<4} | {tc['desc']}")

    print("-" * 80)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_862(Solution862().shortestSubarray)

ID  | 结果     | 预期   | 实际   | 描述
--------------------------------------------------------------------------------
1   | ✅ PASS | 1    | 1    | 1. 示例 1: 基础正数
2   | ✅ PASS | -1   | -1   | 2. 示例 2: 无法满足
3   | ✅ PASS | 3    | 3    | 3. 示例 3: 经典负数回退
4   | ✅ PASS | 1    | 1    | 4. 单个大数直接秒杀
5   | ✅ PASS | 3    | 3    | 5. 剧烈波动的负数
6   | ✅ PASS | -1   | -1   | 6. 全局负数
7   | ✅ PASS | 1    | 1    | 7. 前方大量无用累加
8   | ✅ PASS | 5    | 5    | 8. 连续极长数组带负数断层
--------------------------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

既然有负数，普通的滑动窗口直接报废。我们先求出前缀和数组 `P`。题目转化为：找到 $y > x$，使得 `P[y] - P[x] >= k`，并且让 $y - x$ 最小。

我们维护一个双端队列 `q`，里面存的是前缀和的**索引**。这个队列简直就是一个高度内卷的职场：

1. **规则一：功成身退（左侧收缩）**
   当遍历到当前的 $y$ 时，我们去查队首老大哥 $x$。如果 `P[y] - P[x] >= k`，说明我们找到了一个合法的子数组！
   此时，记录长度 `y - x`。然后**立刻把队首 $x$ 踢出队列**！
   为什么？因为我们要求的是**最短**子数组！即使未来的 $z$ ($z > y$) 也能和 $x$ 满足条件，那长度 $z - x$ 也绝对大于现在的 $y - x$。老大哥 $x$ 的历史使命已经完成，留着没用了。

2. **规则二：降维打击（右侧单调）**
   在新人 $y$ 准备入队时，去看看队尾的家伙 $v$。
   如果 `P[y] <= P[v]`，说明新人 $y$ 的前缀和比老员工 $v$ 还小（或者相等）。
   仔细想想，对于未来的某个 $z$ 来说，想找一个起点相减 `>= k`。选 $y$ 肯定比选 $v$ 更好！因为 $y$ 减出来的差值更大（更容易满足条件），且 $y$ 离 $z$ 更近（子数组更短）！
   所以，$y$ 的出现，让 $v$ 显得毫无价值（又老又弱）。**直接把 $v$ 从队尾踢出去**！这保证了队列里的前缀和永远是**单调递增**的。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。每个前缀和索引最多入队一次，出队一次。虽然代码里有嵌套的 `while` 循环，但全都是常数级的摊还时间。
* **空间复杂度**：$O(N)$。需要额外开辟一个前缀和数组 `P` 以及一个双端队列。
:::